#### এই সপ্তাহে Theory ও Math-এর সম্পর্ক

মানচিত্রে একটা জায়গাকে চেনাতে আমরা দুটো সংখ্যা ব্যবহার করি, অক্ষাংশ ও দ্রাঘিমাংশ। জায়গার নাম না জানলেও ওই দুটো সংখ্যা দিয়ে ঠিক সেই জায়গাটা খুঁজে পাওয়া যায়।

এই সপ্তাহের Theory-তে তুমি পড়বে Tokens, Context Window ও Temperature। সমস্যা হলো, model শব্দ বোঝে না, সে শুধু সংখ্যা বোঝে। তাই লেখাকে ছোট ছোট টুকরো (token) করে প্রতিটা token-কে সংখ্যার একটা তালিকা বানিয়ে model-কে দেওয়া হয়, মানচিত্রের জায়গাকে যেমন সংখ্যা দিয়ে চেনানো হয়।

এখানেই Math কাজে আসে। সংখ্যার এই তালিকার নামই Vector, তার প্রতিটা সংখ্যা একেকটা Scalar, আর তালিকায় কয়টা সংখ্যা আছে সেটাই তার Dimension। এই Vector-ই পরে Embedding হয়ে token-এর অর্থ ধরে রাখে, সেটা আমরা Week 03-এ পড়ব।

> **সারকথা:** Token হলো লেখার টুকরো, আর Vector হলো সেই টুকরোর সংখ্যার রূপ।

#### Part 1: AI Theory

#### Tokenization ও Tokenizer

#### আগে একটা গল্প
দেয়াল তুলতে হলে রাজমিস্ত্রি আগে ইট সাজায়। পুরো দেয়াল এক টুকরো পাথরে কাটা হয় না। ছোট ছোট ইট জুড়েই বড় দেয়াল হয়। Computer লেখা পড়ার সময় ঠিক উল্টোটা করে: বড় বাক্যকে ভেঙে ছোট ছোট ইট বানায়।

#### Token আর Tokenizer
লেখা ভাঙার কাজকে বলে **tokenization**। যে program এই ভাঙার কাজ করে, তার নাম **tokenizer** (লেখা-কাটার যন্ত্র)। ভাঙা প্রতিটা টুকরোর নাম **token**। একটা token পুরো শব্দ হতে পারে, শব্দের অংশ হতে পারে, এমনকি একটা চিহ্নও হতে পারে।

#### নম্বর কেন দরকার?
Computer অক্ষর বোঝে না, সে শুধু সংখ্যা বোঝে। তাই tokenizer প্রতিটা টুকরোকে একটা নম্বর দেয়, যার নাম **token ID**। Model এই নম্বরের সারিই পড়ে। উত্তর দেওয়ার সময়ও সে আগে নম্বর বানায়, পরে tokenizer সেই নম্বর আবার লেখায় ফিরিয়ে দেয়।

> "আমি ভাত খাই" → ["আমি", "ভাত", "খাই"] → [0, 1, 2]  
> লেখা → টুকরো → নম্বর

**Demo:** নিচের code-এ একটা বাক্যকে শব্দে ভেঙে প্রতিটাকে একটা নম্বর দেওয়া হয়েছে, তারপর নম্বর থেকে আবার বাক্য ফেরানো হয়েছে।

#### মনে রাখার এক লাইন
> Tokenizer লেখাকে টুকরো করে নম্বর দেয়, আর model সেই নম্বরই বোঝে।

In [1]:
sentence = "আমি ভাত খাই আমি চা খাই"

tokens = sentence.split()
print("tokens:", tokens)

vocab = {}

for token in tokens:
    if token not in vocab:
        vocab[token] = len(vocab)

print("vocab :", vocab)

token_ids = [vocab[token] for token in tokens]
print("ids   :", token_ids)

id_to_token = {number: token for token, number in vocab.items()}

print("ফেরত  :", " ".join(id_to_token[number] for number in token_ids))

tokens: ['আমি', 'ভাত', 'খাই', 'আমি', 'চা', 'খাই']
vocab : {'আমি': 0, 'ভাত': 1, 'খাই': 2, 'চা': 3}
ids   : [0, 1, 2, 0, 3, 2]
ফেরত  : আমি ভাত খাই আমি চা খাই


#### Tokens vs Words

#### আগে একটা গল্প
বাসে উঠলে ভাড়া কীভাবে ঠিক হয়? যাত্রী কতজন, তা দিয়ে নয়। কত কিলোমিটার গেলে, তা দিয়ে। অনেক দূর গেলে ভাড়া বেশি, কাছে গেলে কম।

AI-র খরচও এভাবে মাপা হয়। মাপার একক শব্দ নয়, **token** (লেখার ছোট টুকরো)।

#### ১ token ≠ ১ শব্দ
অনেকে ভাবে, একটা শব্দ মানেই একটা token। এটা ঠিক নয়। ছোট আর চেনা শব্দ একটা token হতে পারে। লম্বা বা অচেনা শব্দ ভেঙে দুই-তিন token হয়।

ইংরেজিতে গড়ে ১টা শব্দ প্রায় **১.৩ token**। বাংলায় সাধারণত আরও বেশি token লাগে, কারণ বাংলা অক্ষর বেশি টুকরো হয়ে ভাঙে। একই কথা বাংলায় লিখলে তাই token বেশি গোনা হয়।

#### এর ফল কী?
Model-এর **খরচ** আর **সীমা**, দুটোই token দিয়ে মাপা হয়। বাংলায় বেশি token লাগলে খরচও কিছুটা বেশি পড়তে পারে, আর সীমার জায়গাও বেশি খরচ হয়। তাই "কত শব্দ" না ভেবে "কত token" ভাবতে হয়।

> বাসের ভাড়া → কিলোমিটারে মাপা হয়, যাত্রীর সংখ্যায় নয়  
> AI-র খরচ → token-এ মাপা হয়, শব্দের সংখ্যায় নয়

#### মনে রাখার এক লাইন
> খরচ ও সীমা মাপা হয় token দিয়ে। ১ token সবসময় ১ শব্দ নয়, আর বাংলায় সাধারণত বেশি token লাগে।

#### Context Window ও Context Length

#### আগে একটা গল্প
তুমি একটা ছোট টেবিলে পড়তে বসেছ। টেবিলে একসাথে পাঁচটা বই ধরে। ছয় নম্বর বই রাখতে চাইলে একটা পুরনো বই সরাতে হবে। যে বই সরালে, তার কথা আর তোমার সামনে নেই। সেটা তুমি এখন দেখতে পাচ্ছ না।

#### Context window
Model-এরও এমন একটা টেবিল আছে। তার নাম **context window**। এটা হলো model একবারে যতটা লেখা সামনে রেখে কাজ করতে পারে, তার সীমা। এই সীমার মাপকে বলে **context length**, আর তা মাপা হয় token দিয়ে।

এই টেবিলে শুধু তোমার প্রশ্নই থাকে না। আগের কথাবার্তা, তুমি যে লেখা দিয়েছ, আর model-এর নিজের লেখা উত্তরও থাকে।

#### সীমা পেরোলে কী হয়?
কথোপকথন লম্বা হয়ে সীমা পেরোলে সবচেয়ে পুরনো অংশ বাদ পড়ে যায়। তখন model আগের কথা আর মনে রাখতে পারে না, যেন সে ভুলে গেছে।

> context window = [ পুরনো কথা | তোমার প্রশ্ন | model-এর উত্তর ]  
> জায়গা ভরে গেলে → একদম পুরনো অংশ বাদ

#### মনে রাখার এক লাইন
> Context window হলো model-এর টেবিলের জায়গা। জায়গা ভরে গেলে পুরনো কথা বাদ পড়ে।

#### Input Tokens ও Output Tokens

#### আগে একটা গল্প
চা-দোকানে গিয়ে তুমি বললে, "এক কাপ দুধ-চা, কম চিনি।" দোকানদার চা বানিয়ে দিল। এখানে দুটো কাজ হলো: তুমি অর্ডার দিলে, আর সে চা বানাল। দুটো কাজের খাটুনি আলাদা।

#### Input আর Output
LLM-এর সাথে কথা বলাও এমন। তুমি যা পাঠাও, তার নাম **input**, আর তার টুকরোগুলো **input tokens**। Model যা লিখে ফেরত দেয়, তার নাম **output**, আর টুকরোগুলো **output tokens**।

Input-এ থাকে তোমার প্রশ্ন, নির্দেশ আর আগের কথাবার্তা। Output-এ থাকে model-এর লেখা উত্তর।

#### দাম আলাদা কেন?
সাধারণত দুটোর দাম আলাদা, আর output-এর দাম input-এর চেয়ে বেশি। কারণ উত্তর লিখতে model-কে একটা একটা করে token বানাতে হয়। সেটা বেশি খাটুনির কাজ। তাই উত্তর ছোট হলে খরচও কমে।

> তুমি পাঠালে → input tokens (দাম তুলনায় কম)  
> model লিখল → output tokens (দাম তুলনায় বেশি)

#### মনে রাখার এক লাইন
> যা পাঠাও তা input, যা পাও তা output। দুটোর দাম আলাদা, সাধারণত output-এর বেশি।

#### Temperature, Top-p ও Sampling

#### আগে একটা গল্প
তোমার প্রিয় চা-দোকানে তুমি রোজ একই চা খাও। স্বাদ জানা, কোনো ঝুঁকি নেই। কোনোদিন সাহস করে নতুন কিছু চাইলে হয়তো দারুণ কিছু পেলে, হয়তো বিস্বাদ। নিরাপদ আর নতুনের মাঝে কতটা সাহস নেবে, সেটা তোমার হাতে।

#### Sampling
আগের topic-এ দেখেছ, model পরের token-এর probability হিসাব করে। এখন প্রশ্ন, সে কোনটা বাছবে? সবসময় সবচেয়ে সম্ভাব্যটা, নাকি কখনো কখনো অন্যটাও? সম্ভাবনা ধরে লটারির মতো বাছাই করার পদ্ধতির নাম **sampling**।

#### Temperature
**Temperature** একটা নিয়ন্ত্রণ-নব, যা বাছাইয়ে কতটা সাহস থাকবে তা ঠিক করে।

> কম temperature → নিরাপদ, প্রায় একই রকম উত্তর  
> বেশি temperature → সৃজনশীল, কিন্তু এলোমেলো

ভেতরে ব্যাপারটা এমন: model প্রতিটা বিকল্পকে একটা **score** (পছন্দের নম্বর) দেয়। **Softmax** হলো সেই পদ্ধতি, যা score-কে probability-তে বদলায়। Temperature এই বদলানোর সময় বিকল্পগুলোর ফারাক বাড়ায় বা কমায়।

#### Top-p
**Top-p** আরেকটা নিয়ন্ত্রণ। এটা বলে, "শুধু সবচেয়ে সম্ভাব্য কয়েকটা বিকল্পের মধ্য থেকে বাছো, বাকিগুলো বাদ।" যেমন চা-দোকানের পুরো মেনু না দেখে শুধু জনপ্রিয় কয়েকটা আইটেম থেকে অর্ডার করা।

**Demo:** নিচের code-এ একই score-এ temperature 0.5, 1.0 আর 2.0 দিয়ে softmax করে probability দেখানো হয়েছে।

#### মনে রাখার এক লাইন
> Temperature কম হলে নিরাপদ ও একই রকম, বেশি হলে সৃজনশীল ও এলোমেলো। Top-p বাছাইকে সেরা কয়েকটার মধ্যে সীমিত রাখে।

`scaled.max()` বাদ দেওয়া হয় যাতে `np.exp` বড় সংখ্যায় overflow না করে। সব সংখ্যা থেকে একই মান বাদ দিলে অনুপাত বদলায় না, তাই ফলও বদলায় না। শেষে মোট (`exp_values.sum()`) দিয়ে ভাগ করায় সব probability-র যোগফল ১ হয়।

In [2]:
import numpy as np

words = ["খাই", "খাচ্ছি", "রাঁধি", "ঘুমাই"]

scores = np.array([4.0, 3.0, 2.0, 1.0])


def softmax_with_temperature(scores, temperature):
    scaled = scores / temperature
    exp_values = np.exp(scaled - scaled.max())
    return exp_values / exp_values.sum()


for temperature in [0.5, 1.0, 2.0]:
    probs = softmax_with_temperature(scores, temperature)
    print(f"temperature = {temperature}")
    for word, prob in zip(words, probs):
        print(f"  {word:<8} {prob:.3f} {'█' * int(prob * 40)}")

temperature = 0.5
  খাই      0.865 ██████████████████████████████████
  খাচ্ছি   0.117 ████
  রাঁধি    0.016 
  ঘুমাই    0.002 
temperature = 1.0
  খাই      0.644 █████████████████████████
  খাচ্ছি   0.237 █████████
  রাঁধি    0.087 ███
  ঘুমাই    0.032 █
temperature = 2.0
  খাই      0.455 ██████████████████
  খাচ্ছি   0.276 ███████████
  রাঁধি    0.167 ██████
  ঘুমাই    0.102 ████


#### Model Limitations

#### আগে একটা গল্প
তোমার এক বন্ধু আছে, যে প্রচুর বই পড়েছে। প্রায় সব প্রশ্নের উত্তর সে জানে। কিন্তু তার একটা অভ্যাস আছে। যা জানে না, সেটাও সে "জানি না" বলে না। এমন আত্মবিশ্বাসের সাথে একটা গল্প বানিয়ে বলে যে তুমি বিশ্বাস করে ফেলো। সে মিথ্যুক নয়, শুধু ভুল করে। LLM অনেকটা এমনই।

#### পাঁচটা সীমাবদ্ধতা
**১. Hallucination** (না জেনেও আত্মবিশ্বাসের সাথে ভুল বলা)। ভুল তারিখ, বানানো বইয়ের নাম, অচেনা তথ্য, সবই হতে পারে।

**২. Knowledge cutoff** (জ্ঞানের শেষ তারিখ)। Model একটা নির্দিষ্ট তারিখ পর্যন্ত data দিয়ে শিখেছে। তার পরের খবর সে জানে না, যেন অনেকদিন পাহাড়ে কাটিয়ে আসা কেউ।

**৩. Context সীমা।** একবারে সে সীমিত লেখাই সামনে রাখতে পারে। লম্বা কথোপকথনে পুরনো অংশ ভুলে যায়।

**৪. অঙ্ক ও যুক্তিতে ভুল।** সে পরের token অনুমান করে, হিসাব কষে না। তাই বড় অঙ্ক বা জটিল যুক্তিতে ভুল করতে পারে।

**৫. একই প্রশ্নে ভিন্ন উত্তর।** কারণ sampling। প্রতিবার বাছাইয়ে একটু তফাত হয়।

#### তাহলে কী করবে?
LLM-এর উত্তরকে শেষ কথা ধরবে না। গুরুত্বপূর্ণ তথ্য, বিশেষ করে সংখ্যা আর তারিখ, নিজে যাচাই করে নেবে।

#### মনে রাখার এক লাইন
> LLM শক্তিশালী সহযোগী, কিন্তু ভুলহীন নয়। গুরুত্বপূর্ণ কথা নিজে যাচাই করো।

#### Part 2: Math Track

In [3]:
import numpy as np

a = np.array([1, 2, 3])
b = np.array([10, 20, 30, 40])


print(a.shape)
print(b.shape)


(3,)
(4,)


In [4]:
a = np.array([2, 4, 6])
b = np.array([1, 1, 1])

print(a + b)
print(a - b)
print(3 * a)
print(0.5 * a)

[3 5 7]
[1 3 5]
[ 6 12 18]
[1. 2. 3.]


In [5]:
v = np.array(input("enter numbers").split(), dtype=float)

print(np.linalg.norm(v))

5.0


In [6]:
a = np.array([3, 1])
b = np.array([1, 2])

print(a+b)
print(np.linalg.norm(a+b))

[4 3]
5.0
